# WRDNet Training Notebook
**Weather-Resilient Detection Unified Network** — Joint Defogging and Detection with Depth-Guided Feature Fusion

This notebook runs the full WRDNet training pipeline on Google Colab with GPU acceleration.

**Architecture:** DehazeFormer-T (restoration) + YOLOv11s (detection) + FSG/DG-FSG (feature fusion) + Domain Adaptation

**Datasets:** Cityscapes (synthetic fog) + ACDC (real fog) + Foggy Zurich (unlabeled DA) + Foggy Driving (test)

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

In [ ]:
# Clone the repository
!rm -rf /content/object_detection
!git clone https://github.com/soham-kar/object_detection.git /content/object_detection

# Verify clone succeeded
import os
if os.path.exists('/content/object_detection/src'):
    print('Repository cloned successfully.')
else:
    print('ERROR: Clone failed! Check your internet connection.')
    print('Trying again...')
    !git clone https://github.com/soham-kar/object_detection.git /content/object_detection

In [ ]:
# Install dependencies
import os
if not os.path.exists('/content/object_detection'):
    print('ERROR: /content/object_detection not found! Re-run the clone cell above.')
else:
    %cd /content/object_detection
    !pip install ultralytics>=8.3.0 timm>=1.0.27 opencv-python-headless \
        pycocotools tensorboard thop scipy pyyaml tqdm matplotlib seaborn \
        albumentations 2>&1 | tail -5
    print('Dependencies installed.')

In [ ]:
# Clone DehazeFormer (external dependency)
!git clone https://github.com/IDKiro/DehazeFormer.git /content/object_detection/external/DehazeFormer
print('DehazeFormer cloned.')

In [ ]:
# Install DehazeFormer as a package
%cd /content/object_detection/external/DehazeFormer
!pip install -e . 2>&1 | tail -3
%cd /content/object_detection
print('DehazeFormer installed.')

## 2. Mount Google Drive & Load Data

Upload your `data/` folder to Google Drive. The expected structure is:
```
MyDrive/object_detection/data/
  cityscapes/
  rgb_anon_trainvaltest/
  acdc_labels/
  Foggy_Zurich/
  Foggy_Driving/
  depth_stereoscopic_trainvaltest/
  leftImg8bit_trainval_transmittanceDBF/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
import os
import shutil

# Your data is directly in MyDrive/object_detection/ (no data/ subfolder)
DRIVE_DATA = '/content/drive/MyDrive/object_detection'
PROJECT_DATA = '/content/object_detection/data'

# Remove existing data dir and create symlink
if os.path.exists(PROJECT_DATA) and os.path.islink(PROJECT_DATA):
    os.unlink(PROJECT_DATA)
elif os.path.exists(PROJECT_DATA):
    shutil.rmtree(PROJECT_DATA)
os.symlink(DRIVE_DATA, PROJECT_DATA)

# Verify data is accessible
print('Data directories:')
for d in sorted(os.listdir(PROJECT_DATA)):
    full = os.path.join(PROJECT_DATA, d)
    if os.path.isdir(full):
        print(f'  {d}/')
    else:
        print(f'  {d}')

In [ ]:
# Check if labels exist (needed for training)
cityscapes_labels = '/content/object_detection/data/cityscapes/labels'
acdc_labels = '/content/object_detection/data/acdc_labels'

cs_ok = os.path.exists(cityscapes_labels) and len(os.listdir(cityscapes_labels)) > 0
acdc_ok = os.path.exists(acdc_labels) and len(os.listdir(acdc_labels)) > 0

print(f'Cityscapes labels: {"FOUND" if cs_ok else "MISSING"}')
print(f'ACDC labels: {"FOUND" if acdc_ok else "MISSING"}')

if not cs_ok:
    print('\nGenerating Cityscapes labels...')
    !python scripts/convert_cityscapes_labels.py --root data/cityscapes --split train 2>&1 | tail -5
    !python scripts/convert_cityscapes_labels.py --root data/cityscapes --split val 2>&1 | tail -5

if not acdc_ok:
    print('\nGenerating ACDC labels...')
    !python scripts/convert_acdc_labels.py \
        --json data/gt_detection_trainval/gt_detection/fog/instancesonly_fog_train_gt_detection.json \
        --output data/acdc_labels/train 2>&1 | tail -5
    !python scripts/convert_acdc_labels.py \
        --json data/gt_detection_trainval/gt_detection/fog/instancesonly_fog_val_gt_detection.json \
        --output data/acdc_labels/val 2>&1 | tail -5

print('\nLabels ready.')

## 3. Run Label Conversion (if needed)

Skip this section if you already have `cityscapes/labels/` and `acdc_labels/` in your Google Drive.

In [ ]:
# Convert Cityscapes gtFine to YOLO format (run once, ~5 min)
%cd /content/object_detection
!python scripts/convert_cityscapes_labels.py --root data/cityscapes --split train
!python scripts/convert_cityscapes_labels.py --root data/cityscapes --split val

In [ ]:
# Convert ACDC COCO annotations to YOLO format (run once, ~10 sec)
!python scripts/convert_acdc_labels.py \
    --json data/gt_detection_trainval/gt_detection/fog/instancesonly_fog_train_gt_detection.json \
    --output data/acdc_labels/train

!python scripts/convert_acdc_labels.py \
    --json data/gt_detection_trainval/gt_detection/fog/instancesonly_fog_val_gt_detection.json \
    --output data/acdc_labels/val

## 4. Verify Data Pipeline

Run the data pipeline verification to ensure all datasets load correctly.

In [ ]:
%cd /content/object_detection
!python scripts/verify_data_pipeline.py --da

## 5. Smoke Test (Quick Verification)

Run 5 batches through the full training pipeline to verify everything works before committing to a full training run.

In [ ]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

import torch
from src.utils.config import load_config
from src.models.wrnet import WRDNet
from src.training.losses import WRDNetLoss
from src.data.dataset import build_dataloaders

# Load config
config = load_config('configs/default.yaml')
config.batch_size = 2
config.num_workers = 2
config.epochs = 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Build model
print('Building WRDNet...')
model = WRDNet(config).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'  Parameters: {n_params:.2f}M')

# Build loss
criterion = WRDNetLoss(config, yolo_model=model.yolo.model)
print('Loss initialized.')

# Build dataloaders
print('Building dataloaders...')
train_loader, val_loader = build_dataloaders(config)
print(f'  Train: {len(train_loader)} batches')
print(f'  Val: {len(val_loader)} batches')

# Run 5 training batches
print('\nRunning 5 training batches...')
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

for batch_idx, batch in enumerate(train_loader):
    if batch_idx >= 5:
        break
    
    # Move to device
    if 'synth' in batch:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else 
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch['synth'].items()}
        real = {k: v.to(device) if isinstance(v, torch.Tensor) else 
                [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                for k, v in batch['real'].items()}
    else:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else 
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch.items()}
        real = None
    
    # Forward
    outputs = model.forward_train(synth, real)
    
    # Loss
    losses = criterion(outputs, synth)
    
    # Backward
    optimizer.zero_grad()
    losses['total'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    loss_str = ' '.join([f'{k}={v.item():.3f}' for k, v in losses.items() if isinstance(v, torch.Tensor)])
    print(f'  Batch {batch_idx}: {loss_str}')

print('\n✅ Smoke test PASSED! Model can train.')

## 6. Full Training — Phase 0 (Warmup, No Domain Adaptation)

30 epochs of joint defogging + detection training on Cityscapes (synthetic fog).
No domain adaptation in this phase.

In [ ]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

from src.utils.config import load_config
from src.training.trainer import WRDNetTrainer
from src.data.dataset import build_dataloaders

# Load config for Phase 0 (warmup)
config = load_config('configs/default.yaml')

# Adjust for Colab GPU
config.batch_size = 4          # Start with 4, try 8 if no OOM
config.num_workers = 2
config.epochs = 30
config.lr = 1e-3

# No domain adaptation in warmup
config.use_fda = False
config.use_dct_align = False
config.use_fsg_consistency = False

# Checkpointing to Google Drive
config.checkpoint_dir = '/content/drive/MyDrive/object_detection/experiments/checkpoints'
config.log_dir = '/content/drive/MyDrive/object_detection/experiments/logs'

import os
os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.log_dir, exist_ok=True)

print(f'Config: batch_size={config.batch_size}, epochs={config.epochs}, lr={config.lr}')
print(f'Checkpoints: {config.checkpoint_dir}')

# Build dataloaders
print('\nBuilding dataloaders...')
train_loader, val_loader = build_dataloaders(config)

# Create trainer
print('Creating trainer...')
trainer = WRDNetTrainer(config)

# Start training
print('\nStarting Phase 0 training...')
trainer.train(train_loader, val_loader)
print('\nPhase 0 training complete!')

## 7. Full Training — Phase 1 (With Domain Adaptation)

90 epochs with FDA + DCT alignment + FSG consistency.
Pairs synthetic (Cityscapes) with real fog (ACDC + Foggy Zurich).

In [ ]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

from src.utils.config import load_config
from src.training.trainer import WRDNetTrainer
from src.data.dataset import build_dataloaders

# Load config for Phase 1 (domain adaptation)
config = load_config('configs/wrnet_s.yaml')

# Adjust for Colab GPU
config.batch_size = 2    # Smaller batch for DA (paired = 2x memory)
config.num_workers = 2
config.epochs = 90
config.lr = 5e-4

# Enable domain adaptation
config.use_fda = True
config.use_dct_align = True
config.use_fsg_consistency = True
config.real_datasets = ['acdc', 'zurich']

# Checkpointing
config.checkpoint_dir = '/content/drive/MyDrive/object_detection/experiments/checkpoints_da'
config.log_dir = '/content/drive/MyDrive/object_detection/experiments/logs_da'

import os
os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.log_dir, exist_ok=True)

print(f'Config: batch_size={config.batch_size}, epochs={config.epochs}, lr={config.lr}')
print(f'DA: FDA={config.use_fda}, DCT={config.use_dct_align}, FSG={config.use_fsg_consistency}')

# Build dataloaders (will create paired batches)
print('\nBuilding dataloaders...')
train_loader, val_loader = build_dataloaders(config)

# Create trainer (resume from Phase 0 checkpoint)
print('Creating trainer...')
trainer = WRDNetTrainer(config)

# Resume from Phase 0 best checkpoint
phase0_ckpt = '/content/drive/MyDrive/object_detection/experiments/checkpoints/best.pth'
if os.path.exists(phase0_ckpt):
    print(f'Resuming from Phase 0 checkpoint: {phase0_ckpt}')
    trainer.load_checkpoint(phase0_ckpt)
else:
    print('WARNING: No Phase 0 checkpoint found. Starting from scratch.')

# Start training
print('\nStarting Phase 1 training (with DA)...')
trainer.train(train_loader, val_loader)
print('\nPhase 1 training complete!')

## 8. Evaluation on Foggy Driving (Test Set)

Evaluate the trained model on 101 real fog images from Foggy Driving.

In [ ]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

import torch
from src.utils.config import load_config
from src.models.wrnet import WRDNet
from src.data.dataset import build_test_loader
from src.data.collate import wrdnet_collate_fn

config = load_config('configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build model and load checkpoint
model = WRDNet(config).to(device)
checkpoint_path = '/content/drive/MyDrive/object_detection/experiments/checkpoints_da/best.pth'
if not os.path.exists(checkpoint_path):
    checkpoint_path = '/content/drive/MyDrive/object_detection/experiments/checkpoints/best.pth'

ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint: {checkpoint_path}')

# Build test loader
test_loader = build_test_loader(config)

# Run inference
print(f'\nRunning inference on {len(test_loader.dataset)} Foggy Driving images...')
all_detections = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        images = batch['image'].to(device)
        outputs = model(images)
        
        # YOLO output is (raw_preds, decoded_dict)
        det = outputs['detections']
        if isinstance(det, tuple):
            raw = det[0]  # [B, 84, 8400]
        else:
            raw = det
        
        all_detections.append(raw.cpu())
        
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(test_loader)}')

print(f'\nInference complete. {len(all_detections)} batches processed.')
print('TODO: Compute mAP@50 and mAP@50:95 using pycocotools.')

## 9. Visualize Alpha Maps (FSG Interpretability)

Generate alpha map visualizations showing where the FSG trusts defogged vs. original features.

In [ ]:
%cd /content/object_detection

import torch
import matplotlib.pyplot as plt
import numpy as np
from src.utils.config import load_config
from src.models.wrnet import WRDNet
from src.data.foggy_driving import FoggyDrivingDataset
from src.data.collate import wrdnet_collate_fn
from torch.utils.data import DataLoader

config = load_config('configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build model and load checkpoint
model = WRDNet(config).to(device)
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Load a few test images
test_ds = FoggyDrivingDataset(root='data/Foggy_Driving', split='all', input_size=640)
loader = DataLoader(test_ds, batch_size=1, collate_fn=wrdnet_collate_fn)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, batch in enumerate(loader):
    if idx >= 4:
        break
    
    image = batch['image'].to(device)
    with torch.no_grad():
        outputs = model(image, return_alpha=True)
    
    # Get alpha map at P3 scale (highest resolution)
    alpha_p3 = outputs['alpha_maps']['P3'][0, 0].cpu().numpy()
    
    # Denormalize image for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = image[0].cpu().permute(1, 2, 0).numpy()
    img = img * std + mean
    img = np.clip(img, 0, 1)
    
    # Show image
    axes[0, idx].imshow(img)
    axes[0, idx].set_title(f'Foggy Image {idx+1}')
    axes[0, idx].axis('off')
    
    # Show alpha map
    axes[1, idx].imshow(img)
    axes[1, idx].imshow(alpha_p3, cmap='jet', alpha=0.5)
    axes[1, idx].set_title(f'Alpha Map (P3) {idx+1}\nRed=trust defogged, Blue=trust original')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/object_detection/alpha_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Alpha visualization saved to Google Drive.')

## 10. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/object_detection/experiments/logs

## 11. Save Checkpoints Back to Google Drive

If you trained without Google Drive checkpointing, copy results now.

In [ ]:
import shutil
import os

src_dir = '/content/object_detection/experiments'
dst_dir = '/content/driveMyDrive/object_detection/experiments'

if os.path.exists(src_dir):
    os.makedirs(dst_dir, exist_ok=True)
    for item in os.listdir(src_dir):
        s = os.path.join(src_dir, item)
        d = os.path.join(dst_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print(f'Experiments saved to Google Drive: {dst_dir}')
else:
    print('No experiments directory found.')

## Notes

**Runtime Tips:**
- Colab free tier: T4 GPU (16GB), ~12 hour session limit
- Batch size 8 for warmup, 4 for DA (paired batches use 2x memory)
- Checkpoints save directly to Google Drive to survive session disconnects
- Phase 0 (30 epochs): ~3-4 hours on T4
- Phase 1 (90 epochs): ~8-10 hours on T4 (may need session renewal)

**If session disconnects:**
- Reconnect and re-run cells 1-4 (setup + data)
- Then run Phase 1 cell with `trainer.load_checkpoint()` to resume

**Expected Results:**
- Phase 0: Restoration loss should decrease, detection loss should stabilize
- Phase 1: mAP on ACDC val should improve with DA
- Alpha maps should show higher alpha (red) in foggy/distant regions